## STAC/ZARR 2026 OLCI inputs retrieval

This notebook allows to retrieve required inputs to prepare integration of S3 OLCI processor.


## 1. Initialization

In [1]:
# Init environment before running a demo notebook.
from resources.utils import *
from resources.dask_utils import *

init_demo()

satellite='sentinel-3a'
proc_date='2026-04-01'

# Reload the global vars again
from resources.utils import *
from resources.dask_utils import *

14:27:51.020 [DEBUG] (rs_common.prefect_utils) Use API key (probably from '~/.env'): 'e2c0***'


Auxip service: https://rspy.ops.rs-python.eu/auxip
PRIP service: https://rspy.ops.rs-python.eu/prip
CADIP service: https://rspy.ops.rs-python.eu/cadip
Catalog service: https://rspy.ops.rs-python.eu
Staging service: https://rspy.ops.rs-python.eu
DPR service: https://rspy.ops.rs-python.eu
OSAM service: http://rs-server-osam.processing.svc.cluster.local:8080


## 2. Find at least three valid S3A sessions from CADIP as STAC items

In [2]:
sessions = cadip_client.search(method="GET", limit=20, max_items=20, collections="s3_sgs",
                                    stac_filter=f"platform={satellite} and cadip:delivery_push_ok=true",
                                    timestamp=proc_date, sortby=[ { "field": "published", "direction": "desc" } ])
for session in sessions:
  print(f"Session: {session.get_links("self")[0].get_href()}")
assert len(sessions) >= 3

Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401235802052720
Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401221617052719
Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401203510052718
Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401185440052717
Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401171440052716
Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401153459052715
Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401135525052714
Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401121542052713
Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401103536052712
Session: https://rspy.ops.rs-python.eu/cadip/collections/s3_sgs/items/S3A_20260401085459052711
Session: https://rspy.ops.rs-python.eu/cadip/colle

## 3. Find AUX files from ADGS as STAC items

In [3]:
aux_files = []

def process_aux_files(product_type: str, proc_date: str, results: ItemCollection | None):
    if not results:
        raise ValueError(f"No results for {product_type} at {proc_date}")
    for aux in results:
        aux_href = aux.get_links("self")[0].get_href()
        aux_files.append(aux)
        print(f"{product_type}: {aux_href}")

print("-- Common AUX --")
# Query last height to be sure to have at least four on a single date
for product_type in [
    # AUX files common to all satellites where we need at least four on the same date (6 hours each)
    'AX___MA1_AX', 'AX___MA2_AX',  # S00__ADF_ECMWA
    'AX___MF1_AX', 'AX___MF2_AX',  # S00__ADF_ECMWF
    # AUX files common to all satellites (one per type is enough)
    # 'AUX_WND', 'AUX_ECMWFD',                                                                                # S00__ADF_ECMWF
    'AX___DEM_AX',                                                                                            # S00__ADF_GETAS
    # 'AX___BB2_AX', # 'AUX_UT1UTC',                                                                          # S00__ADF_IERSB
    'AX___CLM_AX', 'AX___LWM_AX', 'AX___OOM_AX', 'AX___TRM_AX', 'SR___LSM_AX', 'SR_2_SURFAX', 'SR_2_MLM_AX',  # S00__ADF_WATER
]:
    process_aux_files(product_type, proc_date, auxip_client.search(method="POST", limit=24, max_items=24, stac_filter={
            "op": "and",
            "args": [
                { "op": "=", "args": [ { "property": "product:type" }, product_type ] },
                { "op": "t_intersects", "args": [
                    {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
                    {"interval": [f"{proc_date}T00:00:00.000000Z", f"{proc_date}T23:59:59.000000Z"]}
                ]}
            ]
        }, sortby=[ { "field": "start_datetime", "direction": "desc" } ]))

print(f"-- Satellite-specific AUX for {satellite} --")
# AUX specific per satellite
for product_type in [
    'AX___OSF_AX',  # S03X_ADF_OSFAX
    'AX___FRO_AX',  # S03X_ADF_FROAX
    'AX___FPO_AX',  # S03X_ADF_FPOAX
    'OL_2_ACP_AX',  # S03X_ADF_OLACP
    'OL_1_CAL_AX',  # S03X_ADF_OLCAL
    'OL_2_CLP_AX',  # S03X_ADF_OLCLP
    'OL_1_EO__AX',  # S03X_ADF_OLEOP
    'OL_1_INS_AX',  # S03X_ADF_OLINS
    'OL_1_CLUTAX',  # S03X_ADF_OLLUT
    'OL_2_OCP_AX',  # S03X_ADF_OLOCP
    'OL_2_PCP_AX',  # S03X_ADF_OLPCP
    'OL_2_PPP_AX',  # S03X_ADF_OLPPP
    'OL_1_PRG_AX',  # S03X_ADF_OLPRG
    'OL_1_RAC_AX',  # S03X_ADF_OLRAC
    'OL_1_SPC_AX',  # S03X_ADF_OLSPC
    'OL_2_VGP_AX',  # S03X_ADF_OLVGP
    'OL_2_WVP_AX',  # S03X_ADF_OLWVP
]:
    process_aux_files(product_type, proc_date, auxip_client.search(method="POST", limit=24, max_items=24, stac_filter={
            "op": "and",
            "args": [
                { "op": "=", "args": [ { "property": "product:type" }, product_type ] },
                { "op": "=", "args": [ { "property": "platform" }, satellite ] },
                { "op": "t_intersects", "args": [
                    {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
                    {"interval": [f"{proc_date}T00:00:00.000000Z", f"{proc_date}T23:59:59.000000Z"]}
                ]}
            ]
        }, sortby=[ { "field": "start_datetime", "direction": "desc" } ]))

-- Common AUX --
AX___MA1_AX: https://rspy.ops.rs-python.eu/auxip/collections/S3-AX___MA1_AX/items/S3__AX___MA1_AX_20260401T210000_20260402T090000_20260402T054420___________________ECW_O_SN_001.SEN3
AX___MA1_AX: https://rspy.ops.rs-python.eu/auxip/collections/S3-AX___MA1_AX/items/S3__AX___MA1_AX_20260401T150000_20260402T030000_20260402T053731___________________ECW_O_SN_001.SEN3
AX___MA1_AX: https://rspy.ops.rs-python.eu/auxip/collections/S3-AX___MA1_AX/items/S3__AX___MA1_AX_20260401T090000_20260401T210000_20260401T174832___________________ECW_O_SN_001.SEN3
AX___MA1_AX: https://rspy.ops.rs-python.eu/auxip/collections/S3-AX___MA1_AX/items/S3__AX___MA1_AX_20260401T030000_20260401T150000_20260401T174741___________________ECW_O_SN_001.SEN3
AX___MA1_AX: https://rspy.ops.rs-python.eu/auxip/collections/S3-AX___MA1_AX/items/S3__AX___MA1_AX_20260331T210000_20260401T090000_20260401T054342___________________ECW_O_SN_001.SEN3
AX___MA1_AX: https://rspy.ops.rs-python.eu/auxip/collections/S3-AX___MA1_

## 4. Create Catalog collections

In [4]:
temporal = TemporalExtent([datetime(2026, 1, 1), datetime.now()])

sessions_coll = get_or_create_test_collection(
    collection_id="s3_sessions", description="S3 staged sessions", title="S3 sessions", temporal=temporal)

aux_coll = get_or_create_test_collection(
    collection_id="s3_aux", description="S3 staged auxiliary files", title="S3 auxiliary files", temporal=temporal)

print(f"S3 sessions collection: {sessions_coll.get_links('self')[0].get_href()}")
print(f"S3 aux collection: {aux_coll.get_links('self')[0].get_href()}")
print("STAC Browser: https://stac-browser-catalog.ops.rs-python.eu")

S3 sessions collection: https://rspy.ops.rs-python.eu/catalog/collections/vprivat:s3_sessions
S3 aux collection: https://rspy.ops.rs-python.eu/catalog/collections/vprivat:s3_aux
STAC Browser: https://stac-browser-catalog.ops.rs-python.eu


## 5. Stage sessions in the STAC Catalog

In [5]:
#for session in sessions:
#  stage_single_item(session, sessions_coll)

## 6. Stage aux files in the STAC Catalog

In [6]:
for aux in aux_files:
  stage_single_item(aux, aux_coll)

14:28:04.580 [INFO] (resources.utils) job_status: {'status': 'successful', 'type': 'process', 'created': '2026-04-08T14:28:04Z', 'updated': '2026-04-08T14:28:04Z', 'processID': 'staging', 'progress': 100, 'started': '2026-04-08T14:28:04Z', 'message': 'Finished without processing any tasks', 'jobID': 'b8b5905b-df5b-47d8-a918-50e91562a822'}
14:28:04.580 [INFO] (resources.utils) ----- Staging from 'rspy.ops.rs-python.eu' job 'b8b5905b-df5b-47d8-a918-50e91562a822': SUCCESSFUL 

14:28:04.581 [INFO] (resources.utils) ----- Staging from 'rspy.ops.rs-python.eu' job 'b8b5905b-df5b-47d8-a918-50e91562a822': COMPLETED 

14:28:05.081 [INFO] (rs_client.rs_client) Retrieving specific items from collection 'vprivat:s3_aux'.
14:28:05.582 [INFO] (resources.utils) job_status: {'status': 'successful', 'type': 'process', 'created': '2026-04-08T14:28:05Z', 'updated': '2026-04-08T14:28:05Z', 'processID': 'staging', 'progress': 100, 'started': '2026-04-08T14:28:05Z', 'message': 'Finished without processing an